In [ ]:
# ============================================================
# SignLingo Round 2 -- Phase 0: compact_features
# ============================================================
# Converts the stored (30, 447) landmark arrays into the (30, 146) compact
# 4-block vector the Round 2 pipeline trains on.
#
#   A  hands            126   both hands, 21 points x 3, unchanged
#   B  articulator pose  13   upper-arm and forearm unit vectors x 2 sides,
#                             plus one shoulder-tilt angle
#   C  postural NMS       3   head roll, yaw, pitch
#   D  facial NMS         4   mouth aperture, mouth width, eyebrow raise L/R
#                       ---
#                       146
#
# NO VIDEO IS READ AND MEDIAPIPE NEVER RUNS. This is a pure second pass over
# arrays that already exist, which is the whole point of leaving extraction
# untouched: blocks C and D are angles and ratios, and block B is a set of
# directions, so all three are invariant to the translation and uniform scale
# that shoulder normalisation applies. They can be read straight off the stored
# array. Changing a block costs one re-run of this notebook, not a re-extraction.
#
# Block B deliberately drops limb LENGTH and keeps limb DIRECTION. Measured on
# the four-signer set, the old raw-coordinate form raised linear signer
# recoverability from 65.5% (hands alone) to 94.6%, because elbow and wrist
# positions encode arm length and reach. Nothing is lost by dropping wrist
# position: hand landmark 0 is the wrist and already sits in block A.
#
# This is a copy of derive_frame from the extraction repo's compact_features.py,
# inlined so this project needs nothing from that repo at run time. The self-check
# below is what keeps the two from drifting apart silently.
import os
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np

In [ ]:
# ============================================================
# 0 -- Config
# ============================================================
SRC_DIR = "features"           # the (30, 447) tree, as already extracted
DST_DIR = "features_compact"   # written here, read by prepare_features
FORCE = True                   # re-derive files that already exist

# Face-mesh ids kept by extraction.py. It filters enumerate(face_landmarks), so
# the STORED order is ascending id, not the grouped order written below. Getting
# this backwards is what broke the Round 1 mirror mapping twice.
SELECTED_FACE_IDS = [
    # Lips
    0, 13, 14, 17, 37, 39, 40, 61, 78, 80, 81, 82, 84, 87, 88, 91, 95, 146,
    178, 181, 191, 267, 269, 270, 291, 308, 310, 311, 312, 314, 317, 318,
    321, 324, 375, 402, 405, 415,
    # Eyebrows
    46, 52, 53, 55, 65, 70, 105, 107, 276, 282, 283, 285, 295, 300, 334, 336,
    # Left cheek
    50, 118, 123, 137, 205, 206, 207, 212, 214, 216,
    # Right cheek
    280, 347, 352, 366, 425, 426, 427, 432, 434, 436,
]
assert len(SELECTED_FACE_IDS) == len(set(SELECTED_FACE_IDS)), "duplicate face id"

_FACE_SORTED = sorted(SELECTED_FACE_IDS)
_SLOT = {mid: i for i, mid in enumerate(_FACE_SORTED)}
N_FACE = len(_FACE_SORTED)

# Layout derived from the id list, so changing the face selection moves every
# boundary at once instead of leaving literals to update by hand.
POSE_END = 33 * 3
FACE_END = POSE_END + N_FACE * 3
DIM_RAW = FACE_END + 2 * 21 * 3      # 447 for the current 74-point selection
DIM_COMPACT = 146
SEQUENCE_LENGTH = 30

# Cut points for the four ablation arms. Each arm is a prefix of the vector, so
# an ablation arm is a plain column slice downstream.
ARMS = {"A_hands": 126, "AB_artic": 139, "ABC_postural": 142, "ABCD_facial": 146}

# Eyebrow groups by side. Pose landmark 2 is the left eye, 5 the right, and each
# group below sits consistently on one side (checked against the data).
BROW_LEFT = [_SLOT[i] for i in (276, 282, 283, 285, 295, 300, 334, 336)]
BROW_RIGHT = [_SLOT[i] for i in (46, 52, 53, 55, 65, 70, 105, 107)]
LIP_TOP, LIP_BOTTOM = _SLOT[13], _SLOT[14]
LIP_CORNERS = _SLOT[61], _SLOT[291]
LIMBS = ((11, 13), (13, 15), (12, 14), (14, 16))   # L upper, L fore, R upper, R fore

In [ ]:
# ============================================================
# 1 -- The derivation
# ============================================================
def _unit(v):
    """Direction of v, or zeros when the joint was not detected."""
    n = np.linalg.norm(v)
    return v / n if n > 1e-6 else np.zeros(3)


def derive_frame(frame):
    """One (447,) shoulder-normalized frame -> one (146,) compact frame."""
    pose = frame[0:POSE_END].reshape(33, 3)
    face = frame[POSE_END:FACE_END].reshape(N_FACE, 3)
    out = np.zeros(DIM_COMPACT)

    out[0:126] = frame[FACE_END:DIM_RAW]           # A: left hand then right hand

    # extraction.py stores zeros when a block was not detected. Directions and
    # ratios need real landmarks, so B, C and D stay zero rather than dividing
    # by nothing.
    if not pose.any():
        return out

    # B: articulator DIRECTION, not position. No denominator is needed or wanted:
    # a unit vector is already scale-free, and dividing by an inter-ocular
    # distance would mix a face measurement into an arm measurement.
    for k, (a, b) in enumerate(LIMBS):
        out[126 + 3 * k:129 + 3 * k] = _unit(pose[b] - pose[a])
    # Shoulder tilt. Shoulder normalisation centres on the shoulder midpoint and
    # scales by shoulder distance without rotating, so afterwards the two
    # shoulders sit at +/-0.5 of the shoulder unit vector and their coordinates
    # encode the tilt and nothing else.
    out[138] = np.arctan2(*(pose[11, :2] - pose[12, :2])[::-1])

    inter_ocular = np.linalg.norm(pose[2, :2] - pose[5, :2])
    if inter_ocular < 1e-6:
        return out

    # C: head orientation, from the pose block alone. Roll is an angle and needs
    # no denominator; yaw is an offset over ear distance; pitch is a distance
    # over inter-ocular.
    dx, dy = pose[2, :2] - pose[5, :2]             # right eye -> left eye
    out[139] = np.arctan2(dy, dx)                  # roll
    ear_dist = np.linalg.norm(pose[7, :2] - pose[8, :2])
    if ear_dist > 1e-6:
        out[140] = (pose[0, 0] - (pose[7, 0] + pose[8, 0]) / 2) / ear_dist   # yaw
    # ponytail: pitch off 2D y only. In projection a nod and a lowered head both
    # just move the nose down. If it reads as noise, try pose z, or drop pitch
    # and shift the block D indices down by one.
    out[141] = (pose[0, 1] - (pose[2, 1] + pose[5, 1]) / 2) / inter_ocular   # pitch

    # D: facial NMS, every entry a distance over inter-ocular. y grows downward,
    # so eye_y - brow_y is positive when the brow is raised.
    if not face.any():
        return out
    out[142] = abs(face[LIP_BOTTOM, 1] - face[LIP_TOP, 1]) / inter_ocular
    out[143] = np.linalg.norm(face[LIP_CORNERS[1], :2]
                              - face[LIP_CORNERS[0], :2]) / inter_ocular
    out[144] = (pose[2, 1] - face[BROW_LEFT, 1].mean()) / inter_ocular
    out[145] = (pose[5, 1] - face[BROW_RIGHT, 1].mean()) / inter_ocular
    return out


def derive(seq):
    """(T, 447) -> (T, 146)."""
    return np.stack([derive_frame(f) for f in seq])

In [ ]:
# ============================================================
# 2 -- Self-check
# ============================================================
# Runs before any file is written. It asserts the properties the whole design
# rests on, so a silently wrong derivation cannot reach the training run.
def _self_check():
    rng = np.random.default_rng(0)
    roll = np.deg2rad(15.0)
    frame = np.zeros(DIM_RAW)
    pose = frame[0:POSE_END].reshape(33, 3)
    face = frame[POSE_END:FACE_END].reshape(N_FACE, 3)

    eye_mid = np.array([0.0, -1.2])
    pose[2, :2] = eye_mid + 0.1 * np.array([np.cos(roll), np.sin(roll)])
    pose[5, :2] = eye_mid - 0.1 * np.array([np.cos(roll), np.sin(roll)])
    pose[7, :2], pose[8, :2] = [0.18, -1.2], [-0.18, -1.2]
    pose[0, :2] = [0.036, -1.15]
    pose[11:17] = rng.normal(size=(6, 3))
    face[LIP_TOP, 1], face[LIP_BOTTOM, 1] = -0.90, -0.84
    face[LIP_CORNERS[0], :2], face[LIP_CORNERS[1], :2] = [-0.08, -0.87], [0.08, -0.87]
    face[BROW_LEFT, 1], face[BROW_RIGHT, 1] = -1.35, -1.30
    frame[FACE_END:DIM_RAW] = rng.normal(size=126)

    c = derive_frame(frame)
    assert c.shape == (DIM_COMPACT,)
    assert np.allclose(c[0:126], frame[FACE_END:DIM_RAW]), "block A is not the hands"
    for k, (a, b) in enumerate(LIMBS):
        seg = c[126 + 3 * k:129 + 3 * k]
        assert np.isclose(np.linalg.norm(seg), 1.0), "limb vector is not unit length"
        assert np.allclose(seg, _unit(pose[b] - pose[a]))
    assert np.isclose(c[139], roll), c[139]
    assert np.isclose(c[140], 0.036 / 0.36)
    assert np.isclose(c[141], 0.05 / 0.2)
    assert np.isclose(c[142], 0.06 / 0.2)
    assert np.isclose(c[143], 0.16 / 0.2)

    # The claim blocks B, C and D rest on: they survive any translation and
    # uniform scale, so shoulder normalisation cannot move them.
    moved = derive_frame(frame * 2.5 + 0.3)
    assert np.allclose(moved[126:DIM_COMPACT], c[126:DIM_COMPACT]), \
        "blocks B/C/D are not invariant to translation and scale"

    # The point of the block B change: a longer arm must not move it...
    longer = frame.copy()
    lp = longer[0:POSE_END].reshape(33, 3)
    lp[15] = lp[13] + 1.7 * (lp[15] - lp[13])
    assert np.allclose(derive_frame(longer)[126:DIM_COMPACT], c[126:DIM_COMPACT]), \
        "block B still encodes limb length"
    # ...while a genuine change of arm configuration must still show up.
    bent = frame.copy()
    bp = bent[0:POSE_END].reshape(33, 3)
    bp[15] = bp[13] + np.array([0.4, -0.3, 0.1])
    assert not np.allclose(derive_frame(bent)[129:132], c[129:132]), \
        "block B lost arm motion"

    # Missing detections must not leak a division.
    assert not derive_frame(np.zeros(DIM_RAW)).any(), "blank frame is not blank"
    no_face = frame.copy()
    no_face[POSE_END:FACE_END] = 0
    nf = derive_frame(no_face)
    assert nf[126:142].any() and not nf[142:DIM_COMPACT].any(), \
        "a missing face should zero block D only"

    assert derive(np.stack([frame, frame])).shape == (2, DIM_COMPACT)
    print("Self-check passed:")
    print("  block A is the hands, block B limb vectors are unit length")
    print("  blocks B/C/D survive translation and uniform scale")
    print("  a 70% longer forearm does not move block B; a bent arm does")
    print("  missing pose or face zeroes the right blocks without dividing")


_self_check()

In [ ]:
# ============================================================
# 3 -- Derive every sequence
# ============================================================
assert os.path.isdir(SRC_DIR), (
    SRC_DIR + "/ not found. This reads the (30, %d) arrays already produced by "
    "extraction; it does not read video." % DIM_RAW)

classes = sorted(d for d in os.listdir(SRC_DIR)
                 if os.path.isdir(os.path.join(SRC_DIR, d)))
print("\nDeriving %s/ -> %s/  (%d classes, force=%s)"
      % (SRC_DIR, DST_DIR, len(classes), FORCE))

written = kept = skipped = 0
bad_shapes = []
for action in classes:
    in_dir = os.path.join(SRC_DIR, action)
    out_dir = os.path.join(DST_DIR, action)
    os.makedirs(out_dir, exist_ok=True)
    for name in sorted(f for f in os.listdir(in_dir) if f.endswith(".npy")):
        out_path = os.path.join(out_dir, name)
        if os.path.exists(out_path) and not FORCE:
            kept += 1
            continue
        seq = np.load(os.path.join(in_dir, name))
        if seq.ndim != 2 or seq.shape[1] != DIM_RAW:
            bad_shapes.append((action, name, seq.shape))
            skipped += 1
            continue
        np.save(out_path, derive(seq).astype(np.float32))
        written += 1
print("  %d written, %d kept, %d skipped" % (written, kept, skipped))
if bad_shapes:
    print("  WARNING: %d file(s) were not (T, %d), e.g. %s"
          % (len(bad_shapes), DIM_RAW, bad_shapes[:3]))
if kept and not FORCE:
    print("  Kept files were NOT re-derived. Set FORCE = True after changing a block,")
    print("  or the training run silently mixes old and new feature definitions.")

In [ ]:
# ============================================================
# 4 -- Verify what was written
# ============================================================
out_files = [os.path.join(DST_DIR, c, f)
             for c in sorted(os.listdir(DST_DIR))
             if os.path.isdir(os.path.join(DST_DIR, c))
             for f in os.listdir(os.path.join(DST_DIR, c)) if f.endswith(".npy")]
assert out_files, "nothing was written to " + DST_DIR

shapes = {np.load(p).shape for p in out_files[:200]}
print("\nWrote %d sequences to %s/" % (len(out_files), DST_DIR))
print("  shapes seen in a 200-file sample: %s" % sorted(shapes))
assert shapes == {(SEQUENCE_LENGTH, DIM_COMPACT)}, (
    "mixed or wrong dimensions in %s. A stale tree from an older derivation is "
    "the usual cause; delete it and re-run with FORCE = True." % DST_DIR)

sample = np.stack([np.load(p) for p in out_files[:200]])
assert np.isfinite(sample).all(), "non-finite values in the derived features"
ok = np.array([[f[0:POSE_END].any() for f in np.load(p)] for p in out_files[:200]])
limb_norms = np.linalg.norm(sample[..., 126:138].reshape(-1, 4, 3), axis=-1)
tilt = np.degrees(sample[..., 138][ok])
roll = np.degrees(sample[..., 139][ok])
print("  all finite: yes")
print("  limb vector norms (detected frames): %.4f to %.4f, expected 1.0"
      % (limb_norms[ok.ravel()].min(), limb_norms[ok.ravel()].max()))
print("  shoulder tilt: median %+.1f deg, %.2f%% beyond +/-150 deg"
      % (np.median(tilt), (np.abs(tilt) > 150).mean() * 100))
print("  head roll:     median %+.1f deg, %.2f%% beyond +/-150 deg"
      % (np.median(roll), (np.abs(roll) > 150).mean() * 100))
# Both are arctan2 outputs, so a cluster near +/-180 would mean the angle wraps
# and tiny head or shoulder movements would flip between +pi and -pi.
assert (np.abs(tilt) > 150).mean() < 0.01, "shoulder tilt wraps around +/-180"
assert (np.abs(roll) > 150).mean() < 0.01, "head roll wraps around +/-180"
print("  no arctan2 wraparound in either angle")

with open("compact_features_meta.json", "w") as fh:
    json.dump({"src": SRC_DIR, "dst": DST_DIR, "dim_raw": DIM_RAW,
               "dim_compact": DIM_COMPACT, "n_sequences": len(out_files),
               "n_classes": len(classes), "arms": ARMS,
               "n_face_landmarks": N_FACE}, fh, indent=2)
print("\nWrote compact_features_meta.json")
print("Next: prepare_features.ipynb")